## 导入必要的包

In [ ]:
import numpy as np
import pandas as pd
import matplotlib as plt
import plotly.express as px
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn .metrics import accuracy_score, f1_score ##后面还会导入别的

## 数据探索，清洗与预处理

### bank数据集

In [ ]:
bank=pd.read_csv(r"D:\WeChat Files\wxid_vzlr2hkxrgil22\FileStorage\File\2024-10\2024.10.25\2024.10.25\data\bank.csv")

In [ ]:
bank.head(2)  ##需要的信息是交易金额和工资收入标记，以便于匹配

In [ ]:
display(bank.info())
display(bank.describe().T)

In [ ]:
bank

In [ ]:
bank.drop('交易时间戳',axis=1,inplace=True)

In [ ]:
bank

#### 处理异常值

In [ ]:
px.box(bank,y='交易金额')

In [ ]:
bank = bank[ bank["交易金额"]>0]
avg_val =bank["交易金额"].mean()
std_val =bank["交易金额"].std()
upper_bound = avg_val + 3 * std_val
lower_bound = avg_val - 3 * std_val
drop_index = (bank ["交易金额"] < lower_bound) | (bank["交易金额"] > upper_bound)
bank = bank[~drop_index]
px.box(bank,y="交易金额")

In [ ]:
bank["交易金额_分箱后"]=pd.cut(bank['交易金额'],3,labels=['低','中','高'])
bank=bank[bank['工资收入标记']==1]  ##只读取是工资收入的记录
bank

In [ ]:
result_incomemean=bank.groupby('new_user_id')['交易金额'].mean()

### overdue数据集

In [ ]:
overdue=pd.read_csv(r"D:\WeChat Files\wxid_vzlr2hkxrgil22\FileStorage\File\2024-10\2024.10.25\2024.10.25\data\overdue.csv")

In [ ]:
overdue.head(2)

In [ ]:
overdue=overdue.drop('Unnamed: 0',axis=1)

In [ ]:
print(overdue.info())
print(overdue.describe().T)

### user数据集

In [ ]:
userinfo=pd.read_csv(r"D:\WeChat Files\wxid_vzlr2hkxrgil22\FileStorage\File\2024-10\2024.10.25\2024.10.25\data\userinfo.csv")

In [ ]:
userinfo

In [ ]:
print(userinfo.describe().T)
userinfo.info()

#### 编码

In [ ]:
categorical_columns = ['性别', '职业', '教育程度', '婚姻状态', '户口类型'] 
userinfo = pd.get_dummies(userinfo, columns=categorical_columns, prefix=categorical_columns,dtype=int)

In [ ]:
userinfo.head(2)

### bill数据集

In [ ]:
bill=pd.read_csv(r"D:\WeChat Files\wxid_vzlr2hkxrgil22\FileStorage\File\2024-10\2024.10.25\2024.10.25\data\bill.csv")

In [ ]:
bill.head(2)

In [ ]:
print(bill.describe().T)
bill.info()

In [ ]:
bill.isnull().sum()

### 初步结论与组合原始

形成对数据集的初步结论
1. 数据集不存在缺失值，数据质量高
2. 数据集整体异常值较少，可靠性强
3. 数据集以信贷风险评估为目标，旨在评估个体的风险，数据集包含了个体的基本信息，银行流水，违约情况，历史还款数据等
4. 经过对各个数据集的初步探索和清洗，下一步构造组合数据集进行进一步的评估

In [ ]:
df=userinfo

In [ ]:
df=df.merge(overdue,on='new_user_id',how='left')

In [ ]:
df = df.merge(result_incomemean.to_frame().rename(columns={'交易金额':'平均交易金额'}), 
              on='new_user_id', 
              how='left')

In [ ]:
df

## 进一步编码与特征生成

这时，要对bill数据集进行更加细致的处理，先前我们对其他三份数据集进行了初步的哑变量编码和分组求值，现在进行特征生成部分

#### 生成状态型特征

In [ ]:
# 基础特征计算
bill["欠款金额"] = bill["上期账单金额"] - bill["上期还款金额"]  # 基础欠款情况
bill["还款比率"] = bill["上期还款金额"] / bill["上期账单金额"]  # 基础还款能力
bill["还款比率"] = bill["还款比率"].replace([np.inf, -np.inf], np.nan).fillna(0)

# 风险相关特征
bill["高额使用率"] = (bill["本期账单余额"] / bill["信用卡额度"] > 0.8).astype(int)  # 高使用率一般更高的违约风险
bill["连续欠款"] = (bill["欠款金额"] > 0) & (bill["上期账单金额"] > 0)  # 连续欠款是违约预警信号
bill["还款压力指数"] = (bill["本期账单最低还款额"] / bill["上期还款金额"]).fillna(0)  # 还款压力越大，违约风险越高
bill["额度消耗速度"] = (bill["信用卡额度"] - bill["可用金额"]) / bill["信用卡额度"]  # 额度消耗过快表示财务压力

agg_features = bill.groupby("new_user_id").agg({
    "还款比率": ["mean", "min", "std"],  # 还款稳定性，std越大表示还款行为越不稳定
    "高额使用率": "mean",  # 高额度使用频率
    "连续欠款": "sum",  # 连续欠款次数
    "还款压力指数": ["mean", "max"],  # 平均和最大还款压力
    "额度消耗速度": ["mean", "max"],  # 额度使用行为
    "欠款金额": ["mean", "max", lambda x: (x > 0).sum()]  # 欠款金额统计和欠款次数
}).round(4)

agg_features.columns = [
    "平均还款比率", "最低还款比率", "还款波动性",
    "高额度使用频率",
    "连续欠款次数",
    "平均还款压力", "最大还款压力",
    "平均额度消耗率", "最大额度消耗率",
    "平均欠款金额", "最大欠款金额", "欠款次数"
]
agg_features = agg_features.fillna(agg_features.mean())
agg_features

In [ ]:
bill.head(2)

#### 生成趋势型特征

* 趋势特征与时间序列有关，下面进入时间序列的分析

In [ ]:
bank_train=bank
bank_train['交易时间'].to_timestamp
tmp = bank_train.groupby("new_user_id")[["交易金额","交易时间"]]

In [ ]:
def get_recent_data(df, time_col, n, unit='M'):
    """
    提取时间序列数据中最近n个时间单位的数据
    
    参数:
    df: DataFrame - 输入的数据框
    time_col: str - 时间列的名称
    n: int - 要提取的时间单位数量
    unit: str - 时间单位('D'天,'W'周,'M'月,'Y'年),默认为'M'月
    """

    df[time_col] = pd.to_datetime(df[time_col])  ##转化为时间戳数据类型
    max_date = df[time_col].max()  ##最近的日期
    
    if unit == 'M':
        start_date = max_date - pd.DateOffset(months=n)  ##优化原函数，使用datatime的日期偏置简化函数
    elif unit == 'D':
        start_date = max_date - pd.DateOffset(days=n)
    elif unit == 'W':
        start_date = max_date - pd.DateOffset(weeks=n)
    elif unit == 'Y':
        start_date = max_date - pd.DateOffset(years=n)
    else:
        raise ValueError("不支持的时间单位,请使用 'D','W','M' 或 'Y'")
    
    filtered_df = df[df[time_col] >= start_date]
    return filtered_df


In [ ]:
import datetime
from dateutil.relativedelta import relativedelta
# 计算6个月的数据统计
recent_data_6 = tmp.apply(lambda x: get_recent_data(x, '交易时间', 6, 'M'))
recent_data_6 = recent_data_6.groupby("new_user_id")[["交易金额"]].sum()
recent_data_6.columns = ["6月内工资收入"]
recent_data_6

In [ ]:
# 计算3个月的数据统计
recent_data_3 = tmp.apply(lambda x: get_recent_data(x, '交易时间', 3, 'M'))
recent_data_3 = recent_data_3.groupby("new_user_id")[["交易金额"]].sum()
recent_data_3.columns = ["3月内工资收入"]
recent_data_3 = recent_data_3.reset_index()
recent_data_3

In [ ]:
# 合并数据并计算趋势
trend_df = pd.merge(recent_data_6, recent_data_3, how='left', on=['new_user_id'])
trend_df["半年到3月前工资收入"] = trend_df["6月内工资收入"] - trend_df['3月内工资收入']
trend_df["过去半年内工资增长趋势"] = (trend_df['3月内工资收入'] >= trend_df['半年到3月前工资收入']).astype(int)
trend_df

#### 生成统计型特征

In [ ]:
# 基础统计特征
stats_features = bank.groupby('new_user_id').agg({
    '交易金额': ['max', 'min', 'std', 'mean']  # 收入的基础统计量
}).round(4)

stats_features.columns = [
    '最高月收入', '最低月收入', '收入波动性', '平均月收入'
]


##### 计算月度波动量

In [ ]:
# 创建一个空的DataFrame来存储所有月份的数据
result_df = pd.DataFrame()

# 循环计算每个月的数据
for month in range(1, 7):
    month_data = tmp.apply(lambda x: get_recent_data(x, '交易时间', month, 'M'))
    month_data = month_data.groupby("new_user_id")[["交易金额"]].sum()
    # 重命名列名
    month_data.columns = [f"{month}月内平均收入"]
    # 如果是第一个月，直接赋值给result_df
    if month == 1:
        result_df = month_data
    # 否则，合并数据
    else:
        result_df = result_df.join(month_data)

# 计算月度变化并求标准差
result_df['波动量'] = result_df.apply(
    lambda x: np.std([
        x[f'{i+1}月内平均收入'] - x[f'{i}月内平均收入'] 
        for i in range(1, 6)
    ]), 
    axis=1
)

In [ ]:
result_df=result_df['波动量']
result_df

In [ ]:
stats_features=stats_features.merge(result_df,on='new_user_id',how='left')
stats_features

## 训练前的准备：合并最终数据集与归一化

### 组合数据集

In [ ]:
df_merged = df.merge(stats_features, on='new_user_id', how='left')\
              .merge(trend_df, on='new_user_id', how='left')\
              .merge(agg_features, on='new_user_id', how='left')

In [ ]:
df_merged

## 训练模型

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

In [ ]:
df_all=df_merged
df_all

In [ ]:
X=df_all.drop(['分数','new_user_id','标签'],axis=1)
X=X.apply(lambda x:(x-np.min(x))/(x.max()-x.min()))
X=X.fillna(0)
y_cls=df_all['标签']
y_reg=df_all['分数']
y_reg=(y_reg-y_reg.min())/(y_reg.max()-y_reg.min())

### 基准model编写

In [ ]:
neuro_net_cls=nn.Sequential(
    nn.Linear(X.shape[1],5),
    nn.ReLU(),
    nn.Linear(5,1),
    nn.Sigmoid() 
)
neuro_net_reg=nn.Sequential(
    nn.Linear(X.shape[1],5),
    nn.ReLU(),
    nn.Linear(5,1),
)
#经过多次调整，发现这个模型的效果最好，果然简单的模型往往是最好的

In [ ]:
batch_size=32
epoch=200
# 定义数据集类,一堆初始化函数，len函数，getitem函数
class CreditDataset(Dataset):
    def __init__(self, X, y_cls, y_reg):
        self.X = torch.FloatTensor(X.values)
        self.y_cls = torch.FloatTensor(y_cls.values)
        self.y_reg = torch.FloatTensor(y_reg.values)
        
    def __len__(self):
        return len(self.X)
        
    def __getitem__(self, idx):
        return self.X[idx], self.y_cls[idx], self.y_reg[idx]
# 损失函数
criterion_cls = nn.BCELoss()  # 二分类交叉熵损失
criterion_reg = nn.MSELoss()  # 均方差损失

# 优化器
optimizer_cls = optim.Adam(neuro_net_cls.parameters(), lr=0.001, betas=(0.9, 0.999))
optimizer_reg = optim.Adam(neuro_net_reg.parameters(), lr=0.001, betas=(0.9, 0.999))

# L2正则化
l2_lambda = 0.0001


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, roc_curve
from sklearn.metrics import r2_score, mean_squared_error
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)  # 5折交叉验证

# 初始化列表存储每一折的结果
acc_list, f1_list, precision_list, recall_list, auc_list = [], [], [], [], []
r2_list, pcc_list, mse_list = [], [], []

for fold, (train_idx, test_idx) in enumerate(kf.split(X)):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_cls_train, y_cls_test = y_cls.iloc[train_idx], y_cls.iloc[test_idx]
    y_reg_train, y_reg_test = y_reg.iloc[train_idx], y_reg.iloc[test_idx]
    
    # 创建数据集和数据加载器
    train_dataset = CreditDataset(X_train, y_cls_train, y_reg_train)
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    
    # 训练模型
    for epoch in range(epoch):
        neuro_net_cls.train()
        neuro_net_reg.train()
        for X_batch, y_cls_batch, y_reg_batch in train_loader:
            # 分类任务
            optimizer_cls.zero_grad()
            outputs_cls = neuro_net_cls(X_batch).squeeze()
            loss_cls = criterion_cls(outputs_cls, y_cls_batch)
            loss_cls.backward()
            optimizer_cls.step()
            
            # 回归任务
            optimizer_reg.zero_grad()
            outputs_reg = neuro_net_reg(X_batch).squeeze()
            loss_reg = criterion_reg(outputs_reg, y_reg_batch)
            loss_reg.backward()
            optimizer_reg.step()
    
    # 评估模型
    neuro_net_cls.eval()
    neuro_net_reg.eval()
    with torch.no_grad():
        X_test_tensor = torch.FloatTensor(X_test.values)
        y_cls_pred = neuro_net_cls(X_test_tensor).squeeze().numpy()
        y_reg_pred = neuro_net_reg(X_test_tensor).squeeze().numpy()
        
        # 分类评估
        y_cls_pred_binary = (y_cls_pred > 0.5).astype(int)
        acc = accuracy_score(y_cls_test, y_cls_pred_binary)
        f1 = f1_score(y_cls_test, y_cls_pred_binary)
        precision = precision_score(y_cls_test, y_cls_pred_binary)
        recall = recall_score(y_cls_test, y_cls_pred_binary)
        auc = roc_auc_score(y_cls_test, y_cls_pred)
        
        # 存储分类结果
        acc_list.append(acc)
        f1_list.append(f1)
        precision_list.append(precision)
        recall_list.append(recall)
        auc_list.append(auc)
        
        # 打印分类结果
        print(f"Fold {fold+1} - Classification: Accuracy={acc}, F1={f1}, Precision={precision}, Recall={recall}, AUC={auc}")
        
        # 绘制ROC曲线（仅在第1折）
        if fold == 0:
            fpr, tpr, _ = roc_curve(y_cls_test, y_cls_pred)
            plt.figure()
            plt.plot(fpr, tpr, label='ROC curve')
            plt.xlabel('False Positive Rate')
            plt.ylabel('True Positive Rate')
            plt.title('ROC Curve')
            plt.legend()
            plt.show()
        
        # 回归评估
        r2 = r2_score(y_reg_test, y_reg_pred)
        pcc = np.corrcoef(y_reg_test, y_reg_pred)[0, 1]
        mse = mean_squared_error(y_reg_test, y_reg_pred)
        
        # 存储回归结果
        r2_list.append(r2)
        pcc_list.append(pcc)
        mse_list.append(mse)
        
        # 打印回归结果
        print(f"Fold {fold+1} - Regression: R2={r2}, PCC={pcc}, MSE={mse}")
        
        # 绘制散点图（仅在第1折）
        if fold == 0:
            plt.figure()
            plt.scatter(y_reg_test, y_reg_pred, alpha=0.5)
            plt.xlabel('True Values')
            plt.ylabel('Predictions')
            plt.title('Regression Scatter Plot')
            plt.show()

# 输出五次交叉验证的平均结果
print("Average Results:")
print(f"Classification: Accuracy={np.mean(acc_list)}, F1={np.mean(f1_list)}, Precision={np.mean(precision_list)}, Recall={np.mean(recall_list)}, AUC={np.mean(auc_list)}")
print(f"Regression: R2={np.mean(r2_list)}, PCC={np.mean(pcc_list)}, MSE={np.mean(mse_list)}")

### 稍微复杂一些的model

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)  # 5折交叉验证
neuro_net_cls2=nn.Sequential(
    nn.Linear(X.shape[1],64),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(64,32),
    nn.ReLU(), 
    nn.Dropout(0.2),
    nn.Linear(32,16),
    nn.ReLU(),
    nn.Linear(16,1),
    nn.Sigmoid()
)
neuro_net_reg2=nn.Sequential(
    nn.Linear(X.shape[1],64),
    nn.ReLU(),
    nn.Dropout(0.2), 
    nn.Linear(64,32),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(32,16),
    nn.ReLU(),
    nn.Linear(16,1),
)
# 分类任务优化器
optimizer_cls2 = optim.Adam(neuro_net_cls2.parameters(), lr=0.0005, betas=(0.9, 0.999), weight_decay=0.001)

# 回归任务优化器
optimizer_reg2 = optim.Adam(neuro_net_reg2.parameters(), lr=0.0005, betas=(0.9, 0.999), weight_decay=0.001)

# L2正则化参数
l2_lambda2 = 0.001

# 初始化列表存储每一折的结果
acc_list, f1_list, precision_list, recall_list, auc_list = [], [], [], [], []
r2_list, pcc_list, mse_list = [], [], []

for fold, (train_idx, test_idx) in enumerate(kf.split(X)):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_cls_train, y_cls_test = y_cls.iloc[train_idx], y_cls.iloc[test_idx]
    y_reg_train, y_reg_test = y_reg.iloc[train_idx], y_reg.iloc[test_idx]
    
    # 创建数据集和数据加载器
    train_dataset = CreditDataset(X_train, y_cls_train, y_reg_train)
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    
    # 训练模型
    for epoch in range(epochs):
        neuro_net_cls2.train()
        neuro_net_reg2.train()
        for X_batch, y_cls_batch, y_reg_batch in train_loader:
            # 分类任务
            optimizer_cls2.zero_grad()
            outputs_cls = neuro_net_cls2(X_batch).squeeze()
            loss_cls = criterion_cls(outputs_cls, y_cls_batch)
            loss_cls.backward()
            optimizer_cls2.step()
            
            # 回归任务
            optimizer_reg2.zero_grad()
            outputs_reg = neuro_net_reg2(X_batch).squeeze()
            loss_reg = criterion_reg(outputs_reg, y_reg_batch)
            loss_reg.backward()
            optimizer_reg2.step()
    
    # 评估模型
    neuro_net_cls2.eval()
    neuro_net_reg2.eval()
    with torch.no_grad():
        X_test_tensor = torch.FloatTensor(X_test.values)
        y_cls_pred = neuro_net_cls2(X_test_tensor).squeeze().numpy()
        y_reg_pred = neuro_net_reg2(X_test_tensor).squeeze().numpy()
        
        # 分类评估
        y_cls_pred_binary = (y_cls_pred > 0.5).astype(int)
        acc = accuracy_score(y_cls_test, y_cls_pred_binary)
        f1 = f1_score(y_cls_test, y_cls_pred_binary)
        precision = precision_score(y_cls_test, y_cls_pred_binary)
        recall = recall_score(y_cls_test, y_cls_pred_binary)
        auc = roc_auc_score(y_cls_test, y_cls_pred)
        
        # 存储分类结果
        acc_list.append(acc)
        f1_list.append(f1)
        precision_list.append(precision)
        recall_list.append(recall)
        auc_list.append(auc)
        
        # 打印分类结果
        print(f"Fold {fold+1} - Classification: Accuracy={acc}, F1={f1}, Precision={precision}, Recall={recall}, AUC={auc}")
        
        # 绘制ROC曲线（仅在第1折）
        if fold == 0:
            fpr, tpr, _ = roc_curve(y_cls_test, y_cls_pred)
            plt.figure()
            plt.plot(fpr, tpr, label='ROC curve')
            plt.xlabel('False Positive Rate')
            plt.ylabel('True Positive Rate')
            plt.title('ROC Curve')
            plt.legend()
            plt.show()
        
        # 回归评估
        r2 = r2_score(y_reg_test, y_reg_pred)
        pcc = np.corrcoef(y_reg_test, y_reg_pred)[0, 1]
        mse = mean_squared_error(y_reg_test, y_reg_pred)
        
        # 存储回归结果
        r2_list.append(r2)
        pcc_list.append(pcc)
        mse_list.append(mse)
        
        # 打印回归结果
        print(f"Fold {fold+1} - Regression: R2={r2}, PCC={pcc}, MSE={mse}")
        
        # 绘制散点图（仅在第1折）
        if fold == 0:
            plt.figure()
            plt.scatter(y_reg_test, y_reg_pred, alpha=0.5)
            plt.xlabel('True Values')
            plt.ylabel('Predictions')
            plt.title('Regression Scatter Plot')
            plt.show()

# 输出五次交叉验证的平均结果
print("Average Results:")
print(f"Classification: Accuracy={np.mean(acc_list)}, F1={np.mean(f1_list)}, Precision={np.mean(precision_list)}, Recall={np.mean(recall_list)}, AUC={np.mean(auc_list)}")
print(f"Regression: R2={np.mean(r2_list)}, PCC={np.mean(pcc_list)}, MSE={np.mean(mse_list)}")

### 更更复杂一些的model

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)  # 5折交叉验证
# 分类网络结构优化
neuro_net_cls2 = nn.Sequential(
    nn.Linear(X.shape[1], 128),  # 增加第一层神经元数量以捕获更多特征
    nn.BatchNorm1d(128),  # 添加批标准化提高训练稳定性
    nn.ReLU(),
    nn.Dropout(0.3),  # 略微增加dropout防止过拟合
    
    nn.Linear(128, 64),
    nn.BatchNorm1d(64),
    nn.ReLU(), 
    nn.Dropout(0.3),
    
    nn.Linear(64, 32),
    nn.BatchNorm1d(32), 
    nn.ReLU(),
    nn.Dropout(0.2),
    
    nn.Linear(32, 1),
    nn.Sigmoid()
)

# 回归网络结构优化
neuro_net_reg2 = nn.Sequential(
    nn.Linear(X.shape[1], 128),
    nn.BatchNorm1d(128),
    nn.ReLU(),
    nn.Dropout(0.3),
    
    nn.Linear(128, 64),
    nn.BatchNorm1d(64),
    nn.ReLU(),
    nn.Dropout(0.3),
    
    nn.Linear(64, 32),
    nn.BatchNorm1d(32),
    nn.ReLU(),
    nn.Dropout(0.2),
    
    nn.Linear(32, 1)
)

# 优化器参数调整
optimizer_cls2 = optim.Adam(neuro_net_cls2.parameters(), 
                          lr=0.0003,  # 降低学习率提高稳定性
                          betas=(0.9, 0.999),
                          weight_decay=0.0005)  # 减小L2正则化强度

optimizer_reg2 = optim.Adam(neuro_net_reg2.parameters(),
                          lr=0.0003,
                          betas=(0.9, 0.999), 
                          weight_decay=0.0005)


# 初始化列表存储每一折的结果
acc_list, f1_list, precision_list, recall_list, auc_list = [], [], [], [], []
r2_list, pcc_list, mse_list = [], [], []

for fold, (train_idx, test_idx) in enumerate(kf.split(X)):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_cls_train, y_cls_test = y_cls.iloc[train_idx], y_cls.iloc[test_idx]
    y_reg_train, y_reg_test = y_reg.iloc[train_idx], y_reg.iloc[test_idx]
    
    # 创建数据集和数据加载器
    train_dataset = CreditDataset(X_train, y_cls_train, y_reg_train)
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
    
    # 训练模型
    for epoch in range(epochs):
        neuro_net_cls2.train()
        neuro_net_reg2.train()
        for X_batch, y_cls_batch, y_reg_batch in train_loader:
            # 分类任务
            optimizer_cls2.zero_grad()
            outputs_cls = neuro_net_cls2(X_batch).squeeze()
            loss_cls = criterion_cls(outputs_cls, y_cls_batch)
            loss_cls.backward()
            optimizer_cls2.step()
            
            # 回归任务
            optimizer_reg2.zero_grad()
            outputs_reg = neuro_net_reg2(X_batch).squeeze()
            loss_reg = criterion_reg(outputs_reg, y_reg_batch)
            loss_reg.backward()
            optimizer_reg2.step()
    
    # 评估模型
    neuro_net_cls2.eval()
    neuro_net_reg2.eval()
    with torch.no_grad():
        X_test_tensor = torch.FloatTensor(X_test.values)
        y_cls_pred = neuro_net_cls2(X_test_tensor).squeeze().numpy()
        y_reg_pred = neuro_net_reg2(X_test_tensor).squeeze().numpy()
        
        # 分类评估
        y_cls_pred_binary = (y_cls_pred > 0.5).astype(int)
        acc = accuracy_score(y_cls_test, y_cls_pred_binary)
        f1 = f1_score(y_cls_test, y_cls_pred_binary)
        precision = precision_score(y_cls_test, y_cls_pred_binary)
        recall = recall_score(y_cls_test, y_cls_pred_binary)
        auc = roc_auc_score(y_cls_test, y_cls_pred)
        
        # 存储分类结果
        acc_list.append(acc)
        f1_list.append(f1)
        precision_list.append(precision)
        recall_list.append(recall)
        auc_list.append(auc)
        
        # 打印分类结果
        print(f"Fold {fold+1} - Classification: Accuracy={acc}, F1={f1}, Precision={precision}, Recall={recall}, AUC={auc}")
        
        # 绘制ROC曲线（仅在第1折）
        if fold == 0:
            fpr, tpr, _ = roc_curve(y_cls_test, y_cls_pred)
            plt.figure()
            plt.plot(fpr, tpr, label='ROC curve')
            plt.xlabel('False Positive Rate')
            plt.ylabel('True Positive Rate')
            plt.title('ROC Curve')
            plt.legend()
            plt.show()
        
        # 回归评估
        r2 = r2_score(y_reg_test, y_reg_pred)
        pcc = np.corrcoef(y_reg_test, y_reg_pred)[0, 1]
        mse = mean_squared_error(y_reg_test, y_reg_pred)
        
        # 存储回归结果
        r2_list.append(r2)
        pcc_list.append(pcc)
        mse_list.append(mse)
        
        # 打印回归结果
        print(f"Fold {fold+1} - Regression: R2={r2}, PCC={pcc}, MSE={mse}")
        
        # 绘制散点图（仅在第1折）
        if fold == 0:
            plt.figure()
            plt.scatter(y_reg_test, y_reg_pred, alpha=0.5)
            plt.xlabel('True Values')
            plt.ylabel('Predictions')
            plt.title('Regression Scatter Plot')
            plt.show()

# 输出五次交叉验证的平均结果
print("Average Results:")
print(f"Classification: Accuracy={np.mean(acc_list)}, F1={np.mean(f1_list)}, Precision={np.mean(precision_list)}, Recall={np.mean(recall_list)}, AUC={np.mean(auc_list)}")
print(f"Regression: R2={np.mean(r2_list)}, PCC={np.mean(pcc_list)}, MSE={np.mean(mse_list)}")

发现复杂的模型在分类上表现较为优异，但是回归模型似乎难以有进展
考虑到我的电脑运行一次较为复杂的模型的约莫需要30min。暂时没有进行进一步的模型研究